
# BD Lab - Spark practice


Apache Spark is a lightning-fast cluster computing technology, designed for fast computation. It is based on Hadoop MapReduce and it extends the MapReduce model to efficiently use it for more types of computations, which includes interactive queries and stream processing. The main feature of Spark is its in-memory cluster computing that increases the processing speed of an application.

Spark is designed to cover a wide range of workloads such as batch applications, iterative algorithms, interactive queries and streaming. Apart from supporting all these workload in a respective system, it reduces the management burden of maintaining separate tools.

Apache Spark has following features.

Speed − Spark helps to run an application in Hadoop cluster, up to 100 times faster in memory, and 10 times faster when running on disk. This is possible by reducing number of read/write operations to disk. It stores the intermediate processing data in memory.

Supports multiple languages − Spark provides built-in APIs in Java, Scala, or Python. Therefore, you can write applications in different languages. Spark comes up with 80 high-level operators for interactive querying.

Advanced Analytics − Spark not only supports ‘Map’ and ‘reduce’. It also supports SQL queries, Streaming data, Machine learning (ML), and Graph algorithms.


## Background Spark Context

There are a few Spark concepts that we will now introduce before we get started, specifically the architecture of Spark Applications.

### Spark Applications
Spark Applications consist of a **driver process** and a set of **executor processes**. 
- The **driver process** sits on a node in the cluster and executes your main() function. It is the heart of a Spark Application, responsible for maintaining information about the application, responding to your program or input, and analyzing, distributing, and scheduling work across the executors.
- The **executors** carry out the work assigned by the driver. They execute the code and report the state of the computation back to the driver node.

Spark employs a cluster manager (like YARN, Mesos, or its standalone manager) to keep track of available resources and allocate them to Spark Applications. Note that in **local mode** (which we'll use here), the driver and executors run as threads on your individual computer instead of a cluster.

### The SparkSession and SparkContext
You control your Spark Application through a driver process called the **SparkSession** (or **SparkContext** in older versions/lower-level APIs). 
- **SparkContext** is the entry point to any Spark functionality when working with low-level abstractions like RDDs. Spark applications run as independent sets of processes, coordinated by a Spark Context in a driver program.
- In modern Spark, **SparkSession** is the unified entrance point for reading data and executing user-defined manipulations across the cluster using higher-level APIs (DataFrames/Datasets). 

*Quick Note*: The Spark Context may be automatically created (for instance, if you call pyspark from the shell, it's called `sc`). While modern Spark focuses on SparkSession, using the Spark Context here gives you familiarity with both.

### Partitions
To allow every executor to perform work in parallel, Spark breaks up the data into chunks called **partitions**. A partition is a collection of rows that sits on one physical machine in your cluster. If you have many partitions but only one executor (e.g., local mode), Spark still executes them sequentially or as threads on that computation resource. With DataFrames, you generally do not manipulate partitions manually; Spark dynamically determines how work executes. With RDDs (introduced next), you get more fine-grained control.

### DataFrames
A **DataFrame** is the most common Structured API and represents a table of data with rows and columns (a schema). Unlike a spreadsheet or a Pandas DataFrame that sits on one machine, a Spark DataFrame can span thousands of computers. When run on a cluster, each part (partition) of the data exists on a different executor.

### Transformations, Actions, and Lazy Evaluation
In Spark, core data structures are immutable—they cannot be changed after creation.
- **Transformations**: To "change" data, you instruct Spark how to modify it. These instructions are called transformations (e.g., finding even numbers). There are *narrow* transformations (each input partition contributes to one output partition) and *wide* transformations (input partitions contribute to many output partitions, requiring a "shuffle" across the cluster).
- **Lazy Evaluation**: Spark will wait until the very last moment to execute the computation. When you specify a transformation, Spark simply builds a plan. It waits to compile and optimize the entire data flow until an action is called.
- **Actions**: To trigger computation, you run an action (e.g., `count`, `collect`). An action instructs Spark to compute a result from the series of transformations and either return it to the console, bringing it to a native object, or writing it to storage.


![Title](https://annefou.github.io/pyspark/slides/images/SparkRuntime.png)


## Background on RDD

Resilient Distributed Datasets (RDDs) are a fundamental, low-level data structure in Apache Spark. An RDD represents an immutable, partitioned collection of records that can be operated on in parallel. Unlike DataFrames where records are structured rows, records in RDDs are raw Java, Scala, or Python objects of your choosing.

Internally, each RDD is characterized by:
- A list of partitions
- A function for computing each split
- A list of dependencies on other RDDs
- Optionally, a Partitioner for key-value RDDs (to specify partitioning)
- Optionally, a list of preferred locations on which to compute each split

### When to Use RDDs
While virtually all Spark code compiles down to RDDs under the hood, higher-level Structured APIs (like DataFrames or Datasets) are generally recommended for most use cases because they offer built-in optimizations. However, you should use the lower-level RDD API when:
- You need fine-grained control over physical data placement across the cluster, such as custom partitioning.
- You need to maintain a legacy codebase written using RDDs.
- You need to manipulate custom distributed shared variables (like broadcast variables).

RDDs grant you complete control, but optimizations must be coded manually. Additionally, using RDDs in Python incurs performance overhead since data must be serialized to the Python process, operated on row by row, and then serialized back to the JVM.

### Transformations, Actions, and Lazy Evaluation
RDDs follow the Spark programming paradigm of separating operations into transformations and actions.
- **Transformations** creating a new RDD from an existing one (e.g., map, filter). They evaluate lazily.
- **Actions** instruct Spark to perform computation and send the result back to the driver (e.g., collect, count).

**Lazy Evaluation**: Spark delays execution until an action is invoked. When a transformation is called, Spark simply records the lineage (the graph of operations). Once an action triggers execution, Spark looks at the whole lineage to create an optimized physical execution plan.

## Creating RDDs

There are primarily two ways to create RDDs: by parallelizing an existing collection in your driver program, or by referencing a dataset in an external storage system.

We will start with parallelizing a simple range of numbers to explore RDD transformations and actions. Then, we will load an external CSV file (the Wikipedia movie plots file we have been working with) to be used for word count exercises later on.


First we import pyspark which creates the spark context 'sc' by default we will be working with. (this step is not required in Databricks but we can do so anyway).

## Installing Java and Pyspark (RUN ONCE)

In [ ]:
#Checking the installed Java version
!java -version

In [ ]:
!pip install pyspark

In [ ]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless


In [ ]:
!java -version

In [ ]:

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("PySpark RDDs") \
        .getOrCreate()

In [ ]:
sc = spark.sparkContext

In [ ]:
import pyspark
from pyspark.context import SparkContext

## RDD Manipulations
You manipulate RDDs in much the same way that you manipulate DataFrames, but the core difference is that you manipulate raw Java, Scala, or Python objects instead of Spark types. You must define each filtering, mapping, or aggregation manually as a function. You specify transformations on one RDD to create another.

### Simple Transformations and Actions
Let's create a simple RDD using a Python range and apply some basic transformations like `map` and `filter`, followed by a `collect` action.

In [ ]:
range_data = range(0, 10)
simpleRDD = sc.parallelize(range_data, 2)


We can use map to map the data using a simple function that squares each number, and filter to keep only even numbers.

In [ ]:
# map transformation: squaring data
squaredRDD = simpleRDD.map(lambda x: x**2)

In [ ]:

# filter transformation: only retrieving even records
evenSquaredRDD = squaredRDD.filter(lambda x: x % 2 == 0)

# collect action: retrieving evaluated collection back to the driver
evenSquaredRDD.collect()

Here, we don't actually need to use the `collect` action to see the results, but we will do so here for demonstration purposes. In practice, you should be cautious when using `collect` on large datasets as it brings all data to the driver. Furthermore, Spark's lazy evaluation means that transformations are not executed until an action is called, so the `collect` action triggers the execution of the transformations we defined, i.e, the squared and evensquaredRDDs are not computed until we call `collect()`.

### Caching, Parallelization, and Partitioning
- **Caching (and Checkpointing)**: Because of lazy evaluation, applying transformations repetitively on the same RDD can be computationally expensive. Caching allows you to persist intermediate data of an RDD in memory (or disk via checkpointing), so subsequent references don't recompute the entire lineage. It is essential for iterative tasks like machine learning pipelines.
- **Parallelization & Partitioning**: Under the hood, an RDD is logically split into partitions. Parallelizing tasks means dividing them into components to compute on different worker nodes simultaneously. Proper partitioning is vital; for instance, applying a custom partitioner for key-value RDDs ensures that data sets with identical keys cluster on identically placed nodes. This significantly lowers expensive `OutOfMemoryErrors` and limits cluster network shuffles.

## Loading the Wikipedia Movie Plots Dataset
Now let's replace the synthetic range data with real data. We will rely on evaluating the Wikipedia movie plots CSV file from week01. Each row in this unstructured dataset represents a movie, roughly split down comma-separated indexes as follows: `[Release Year, Title, Origin/Ethnicity, Director, Cast, Genre, Wiki Page, Plot]`. We can use `take` to inspect the first few rows of the dataset.

In [ ]:
wikiRDD = sc.textFile("wiki_movie_plots_deduped.csv")
wikiRDD.take(3)

### Demonstrating Lazy Evaluation and Caching ("The Slow Loading")
Let's perform a relatively expensive operation: parsing the CSV and counting the records. We'll measure the time it takes. Without caching, Spark has to re-read the file and re-parse everything every time we call an action like `count()`.

In [ ]:
import time
import csv

def parse_csv_line(line):
    reader = csv.reader([line])
    return next(reader, [])

header = wikiRDD.first()
# Create an RDD that has to parse every line
parsedMoviesRDD = wikiRDD.filter(lambda row: row != header).map(parse_csv_line)

# Simulate "Slow Loading" without cache
start_time = time.time()
print("Count without cache:", parsedMoviesRDD.count())
print("Time taken (No Cache): {:.2f} seconds".format(time.time() - start_time))

Now, let's `cache()` the RDD. The first time we run an action after calling `cache()`, Spark will compute the RDD and hold it in memory.

In [ ]:
parsedMoviesRDD.cache()

# First action after caching will still take time because it's loading into memory
start_time = time.time()
print("Count during cache:", parsedMoviesRDD.count())
print("Time taken (First time with cache): {:.2f} seconds".format(time.time() - start_time))

Let's run the exact same `count()` action again. Notice how much faster it is! This is because the RDD is now being read directly from the memory cache rather than from disk and being re-parsed. This "slow loading" initially is normal, but subsequent operations are lightning fast.

In [ ]:
start_time = time.time()
print("Count from cache:", parsedMoviesRDD.count())
print("Time taken (Second time with cache): {:.2f} seconds".format(time.time() - start_time))

### Demonstrating Parallelization and Partitioning
You can check how many partitions your RDD is split into. More partitions mean more parallelization (up to the number of cores in your cluster). You can use `repartition()` to increase partitions or `coalesce()` to reduce them efficiently.

In [ ]:
print("Original Partitions:", parsedMoviesRDD.getNumPartitions())

# Increase partitions (forces a full shuffle)
repartitionedRDD = parsedMoviesRDD.repartition(4)
print("Partitions after repartition(4):", repartitionedRDD.getNumPartitions())

# Decrease partitions efficiently without a full shuffle
coalescedRDD = repartitionedRDD.coalesce(2)
print("Partitions after coalesce(2):", coalescedRDD.getNumPartitions())

### Simple Word Count on Titles
Let's employ a familiar Map-Reduce algorithm approach on the `Title` column of this dataset. The basic logic mirrors how Hadoop processed workloads. We will map rows into words, combine key values, and resolve with aggregations.

In [ ]:
import csv
import string

# Helper python method for parsing complexly escaped CSV columns
def parse_csv_line(line):
    # TODO something
    return # Something

# Drop header
header = # TODO something
moviesRDD = # TODO something
# Extract just the Title explicitly (index 1)
titlesRDD = # TODO something

# Apply cleansing: split into words, remove punctuation, shift lowercase, clear empties
wordsRDD = # TODO something

# Assign Map-reduce pairs: (word, 1) and combine!
titleWordCountRDD = # TODO something

# Unearth top 10 title word distributions
titleWordCountRDD

## Common RDD Transformations & Actions
Below are detailed usage applications of a variety of fundamental RDD manipulation commands. We will test these strictly relying on the Wikipedia plot dataset we instantiated above.

**distinct**: Returns a new RDD after dropping duplicate elements within it.
**filter**: Used like a SQL `WHERE` clause. It will search individual records within the RDD and evaluate against any predicate lambda logic that returns a Boolean condition boolean.

In [ ]:
# distinct: Let's see exactly how many unique release years actually possess entries in the file
yearsCount = moviesRDD.map(lambda row: row[0]).distinct().count()
print("Distinct release years:", yearsCount)

# filter: Filter out exclusively recent movies made in 2017
movies2017 = moviesRDD.filter(lambda row: row[0] == '2017')
print("Count of 2017 movies:", movies2017.count())

**map**: Produces a new value record for each separate single input given to it yielding 1-to-1 data ratios.
**flatMap**: Similar to extending map, mapping requires each returned variable to resolve into an iterable. That sequence itself will become flattened into multiple isolated rows (e.g. 1-to-N relationships). Highly functional for tearing sequences apart cleanly (such as strings split into individual tokens).

In [ ]:
# map: Map solely the combination pair consisting of Title and Director exclusively out of 2017 movies
titleDirectorRDD = movies2017.map(lambda row: (row[1], row[3]))
print("Title & Director pairs (first 3):", titleDirectorRDD.take(3))

# flatMap: Rip apart the genre definitions per movie, unpacking arrays of elements cleanly.
# (Genres are normally separated via explicit commas depending on string lengths)
genresFlatRDD = moviesRDD.filter(lambda row: len(row) > 5).flatMap(lambda row: row[5].split(','))
print("Flattened genre extractions:", genresFlatRDD.take(5))

**reduceByKey**: Replaces traditional flat variables via an associative operator acting locally inside isolated node partitions strictly before traversing cross-nodes. Inherently highly preferred over standard implementations involving `groupByKey` due to it reliably averting catastrophic array memory overloads.
**sortBy / sort**: Evaluates an object subset by applying evaluation rules determining positional ordering.

In [ ]:
# Count independent combinations summing total associated usages per key, ordering final values.
genreCounts = genresFlatRDD.map(lambda g: (g.strip().lower(), 1)).reduceByKey(lambda x, y: x + y)
genreCounts.sortBy(lambda current: current[1], ascending=False).take(10)

**collect**: Returns all the elements of the dataset as an array at the driver program. Use with caution on large datasets as it can run out of memory.
**take(n)**: Returns an array with the first n elements of the dataset.
**count**: Returns the number of elements in the dataset.
**first**: Returns the first element of the dataset (similar to take(1)).
**reduce**: Aggregates the elements of the dataset using a function (which takes two arguments and returns one).

In [ ]:
# collect: carefully collect a very small subset
smallSubsetRDD = moviesRDD.map(lambda row: row[1]).sample(False, 0.001)
print("Collect subset length:", len(smallSubsetRDD.collect()))

# count and first
print("Total number of movies:", moviesRDD.count())
print("First movie title:", moviesRDD.map(lambda row: row[1]).first())

# reduce: calculate the total length of all movie titles combined
titleLengthsRDD = moviesRDD.map(lambda row: len(row[1]))
total_length = titleLengthsRDD.reduce(lambda a, b: a + b)
print("Total characters in all movie titles:", total_length)

**union**: Returns a new dataset that contains the union of the elements in the source dataset and the argument.
**intersection**: Returns a new RDD that contains the intersection of elements in the source dataset and the argument.
**groupByKey**: When called on a dataset of (K, V) pairs, returns a dataset of (K, Iterable<V>) pairs. Less efficient than `reduceByKey` for aggregation.
**join**: When called on datasets of type (K, V) and (K, W), returns a dataset of (K, (V, W)) pairs with all pairs of elements for each key.

In [ ]:
# union & intersection
comedyMovies = moviesRDD.filter(lambda row: len(row) > 5 and 'comedy' in row[5].lower()).map(lambda row: row[1])
actionMovies = moviesRDD.filter(lambda row: len(row) > 5 and 'action' in row[5].lower()).map(lambda row: row[1])

actionComedyUnion = comedyMovies.union(actionMovies)
actionComedyIntersection = comedyMovies.intersection(actionMovies)

print("Total Comedy OR Action movies:", actionComedyUnion.count())
print("Movies that are BOTH Comedy AND Action:", actionComedyIntersection.count())

# groupByKey: Group movie titles by their release year (using only a small subset of years to save memory)
movies1920s = moviesRDD.filter(lambda row: len(row) > 0 and row[0].startswith('1920'))
groupedByYear = movies1920s.map(lambda row: (row[0], row[1]) if len(row) > 1 else (row[0], "")).groupByKey()
print("Movies grouped by year:", [(k, list(v)[:3]) for k, v in groupedByYear.take(1)])

# join: Create two dummy RDDs from our subset
rdd1 = sc.parallelize([("1920", "Movie A"), ("1920", "Movie B")])
rdd2 = sc.parallelize([("1920", "1 million USD"), ("1921", "2 million USD")])
joinedRDD = rdd1.join(rdd2)
print("Joined RDD:", joinedRDD.collect())

**sample**: Sample a fraction of the data, with or without replacement, using a given random number generator seed.
**keys** / **values**: For an RDD of key-value pairs `(K, V)`, these transformations return an RDD of just the keys `K` or just the values `V`.
**subtract**: Return an RDD with the elements from the source RDD that are not in the other RDD.
**cartesian**: When called on datasets of types `T` and `U`, returns a dataset of `(T, U)` pairs (all pairs of elements). Use very carefully on large datasets as it results in `O(N*M)` records.

In [ ]:
# sample
sampledRDD = moviesRDD.sample(withReplacement=False, fraction=0.01, seed=42)
print("Sampled count:", sampledRDD.count())

# keys and values
kvRDD = sc.parallelize([("Alice", 25), ("Bob", 30), ("Charlie", 35)])
print("Keys:", kvRDD.keys().collect())
print("Values:", kvRDD.values().collect())

# subtract
justComedy = actionComedyUnion.subtract(actionComedyIntersection)
print("Movies strictly in one genre (Comedy OR Action, not BOTH):", justComedy.count())

# cartesian (on tiny datasets!)
tiny1 = sc.parallelize(["A", "B"])
tiny2 = sc.parallelize(["1", "2"])
print("Cartesian Product:", tiny1.cartesian(tiny2).collect())

## Exercise: Modern Genre Analysis
**Dataset Structure Reminder:** Each row in `moviesRDD` maps to `[Release Year, Title, Origin/Ethnicity, Director, Cast, Genre, Wiki Page, Plot]`.

**Task:**
Using **only** the common core RDD transformations (`filter`, `flatMap`, `map`, `reduceByKey`, `sortBy`) and actions (`take`), find the **top 5 most common single genres for American movies released from the year 2000 onwards**. 

**Steps to follow:**
1. **Filter** `moviesRDD` for `Origin/Ethnicity` (Index `2`) equal to `'American'` (ignoring casing) and `Release Year` (Index `0`) `>= 2000`.
2. **FlatMap** the `Genre` column (Index `5`) splitting by comma (e.g., `"action, comedy" -> ["action", " comedy"]`).
3. **Map** the genres to lower case, strip any whitespace, and emit a `(genre, 1)` pair. Be sure to filter out any empty strings `''` or the string `'unknown'`.
4. **ReduceByKey** to sum the genre frequencies.
5. **SortBy** the counts in descending order.
6. **Take** and print the top 5 results.

In [ ]:
# --- EXERCISE SOLUTION ---

# 1. Filter for American movies from 2000 onwards 
americanModern = # TODO something

# 2. Extract and physically split the Genre column 
# (Many Wikipedia genres are dynamically comma-separated strings)
splitGenres = # TODO something

# 3. Clean string artifacts, map directly out into (genre, 1) associative pairs, and filter blank/unknown traits
cleanedPairs = # TODO something

# 4. Reduce inherently by identical key to sum occurrence totals across partitions safely
genreFrequencies = # TODO something

# 5. Sort final results by occurrences resolving descending values
sortedGenres = # TODO something

# 6. Take top 5 and visually evaluate
top5AmericanModernGenres = # TODO something

print("Top 5 American Modern Genres:")
for genre, count in top5AmericanModernGenres:
    print(f"- {genre.capitalize()}: {count}")

## Complex RDD Operations
Let's explore some more complex ways of interacting with RDDs, primarily around manipulating partitions directly or integrating with system commands.

**pipe**: Return an RDD created by piping elements to a forked external process. The resulting RDD is computed by executing the given process once per partition. All elements of each input partition are written to a process’s stdin.
**mapPartitions**: Operate on a per-partition basis instead of a per-row basis. This is valuable for performing something on an entire sub-dataset of your RDD, like gathering all values of a partition group.
**mapPartitionsWithIndex**: Similar to `mapPartitions` but provides an index for the partition within the cluster.
**foreachPartition**: Iterates over all partitions of the data but has no return value. This makes it great for doing something with each partition like writing it out to a database or creating custom file sources.

In [ ]:
# Use `.pipe` to run a system command like `wc -l` (word count - lines). 
# We'll use a tiny RDD for stability so it doesn't take forever.
tinyWords = sc.parallelize(["Spark", "The", "Definitive", "Guide", "Big", "Data"], 2)
print("Pipe result:", tinyWords.pipe("wc -l").collect())

# mapPartitions: return number 1 for each completely processed partition.
print("Total Partitions:", tinyWords.mapPartitions(lambda part: [1]).sum())

# mapPartitionsWithIndex: Check what items reside in what index block.
def indexedFunc(partitionIndex, withinPartIterator):
    return [f"partition: {partitionIndex} => {x}" for x in withinPartIterator]

print("Indexed Partitions:", tinyWords.mapPartitionsWithIndex(indexedFunc).collect())

# foreachPartition: Let's create a dummy text file to a temporary directory
def saveToTemp(iter):
    import random
    with open(f"/tmp/random-file-{random.randint(1, 100000)}.txt", "w") as fw:
        for item in iter:
            fw.write(f"{item}\n")

tinyWords.foreachPartition(saveToTemp)
print("foreachPartition executed! Saved to /tmp/random-file-*.txt")

## Key-Value RDDs
Many methods on RDDs require putting data into a key-value format (denoted by **ByKey** methods). Spark provides unique functionality when operating across PairRDDs.

**keyBy**: Define a custom lambda function that evaluates across the current value and generates a designated key.
**mapValues** / **flatMapValues**: Explicitly maps over the values (the 2nd element of a Pair tuple) modifying them while keeping the individual keys completely unmodified and stable.
**lookup**: Look up the result for a particular key and fetch both values associated with it. Unlike a `dict`, this can return multiple responses because keys are not enforced as single unique entries in RDDs.
**sampleByKey**: Sample an RDD by a specific set of keys. Use `sampleByKeyExact` to make additional passes over the RDD for a more rigorously precise sample threshold.

In [ ]:
# keyBy: Extract the first letter as a key
keywordRDD = tinyWords.keyBy(lambda word: word.lower()[0])
print("keyBy:", keywordRDD.collect())

# mapValues / flatMapValues
print("mapValues:", keywordRDD.mapValues(lambda word: word.upper()).collect())
print("flatMapValues (first 2):", keywordRDD.flatMapValues(lambda word: list(word.upper())).take(2))

# lookup (looking up 'd')
# We have "Definitive" and "Data" both starting with 'd'
print("lookup 'd':", keywordRDD.lookup("d"))

# sampleByKey
import random
distinctChars = tinyWords.flatMap(lambda word: list(word.lower())).distinct().collect()
sampleMap = dict(map(lambda c: (c, 0.5), distinctChars)) # Attempt roughly 50% fraction randomly
sampledByKeyResult = keywordRDD.sampleByKey(True, sampleMap, 6).collect()
print("sampleByKey (size and actual returns will vary by seed):", sampledByKeyResult)

## Advanced Aggregations
We briefly looked at `reduceByKey` vs `groupByKey`. The implementation is incredibly critical at scale. `groupByKey` holds all values for a given key inside memory before functioning across them, easily triggering OutOfMemoryErrors upon skew. `reduceByKey` operates optimally by pushing a primary map-reduce summation layer locally within the exact nodes prior to clustering across the final driver. In associative tasks (where sequence ordering doesn't restrict addition properties), `reduceByKey` is substantially safer. 

There are further low-level APIs to enforce specific architectural performance handling.

**countByKey**: Counts the number of elements for each existing key directly collecting outputs to a local mapped Dictionary.
**aggregate**: Requires a null (start) value, an intra-partition aggregation function, and a cross-partition aggregation function. All executed actions are brought locally down to the driver machine.
**treeAggregate**: Pushes down the aggregations by creating a multi-level execution tree acting executor-to-executor completely saving driver arrays from extreme memory overflows compared to standard `aggregate()`.
**aggregateByKey**: Like aggregate, but maps groupings down by matching identical keys instead of completely partitioning.
**combineByKey**: Specify a standalone combiner acting over groups of identical keys, then provide merge functions uniting the differing combiner definitions together. 
**foldByKey**: Merges the defined values locally connected to keys via evaluating an associative method mapped linearly against a completely neutral zero-value.

In [ ]:
# Let's create an arbitrary character-value dataset
chars = tinyWords.flatMap(lambda word: word.lower())
KVcharacters = chars.map(lambda letter: (letter, 1))

def maxFunc(left, right):
    return max(left, right)

def addFunc(left, right):
    return left + right

# countByKey
print("countByKey:", KVcharacters.countByKey())

# aggregate & treeAggregate
nums = sc.parallelize(range(1,31), 5)
print("aggregate:", nums.aggregate(0, maxFunc, addFunc))
depth = 3
print("treeAggregate:", nums.treeAggregate(0, maxFunc, addFunc, depth))

# aggregateByKey (starting with 0, adding values within partition, taking max across partitions)
print("aggregateByKey:", KVcharacters.aggregateByKey(0, addFunc, maxFunc).collect())

# combineByKey
def valToCombiner(value):
    return [value]
def mergeValuesFunc(vals, valToAppend):
    vals.append(valToAppend)
    return vals
def mergeCombinerFunc(vals1, vals2):
    return vals1 + vals2

outputPartitions = 6
print("combineByKey:", KVcharacters.combineByKey(valToCombiner, mergeValuesFunc, mergeCombinerFunc, outputPartitions).collect())

# foldByKey (associative fold across 0)
print("foldByKey:", KVcharacters.foldByKey(0, addFunc).collect())

## Advanced Joins & Zips
Like earlier demonstrated, PairRDDs contain identical internal joining structures to traditional SQL processing, returning values mapping directly to specified key conditions. `join` executes inner joins. `fullOuterJoin`, `leftOuterJoin`, and `rightOuterJoin` enforce standard relational matching keeping disjointed indexes mapped against `None` references. Be extremely careful deploying `cartesian` operations absent of join keys, causing extreme factorial `O(N * M)` sizes. 

**zip**: Zipping technically differs from strictly structured joined queries, explicitly functioning by pairing indexed lines uniformly across identical structured independent groupings. Both participating sequences *MUST* contain identically scaled parallel partitions holding equal item lengths perfectly.

In [ ]:
# zip example
# Create a numerical range identical in size/partitions to tinyWords
numRange = sc.parallelize(range(6), 2)
zippedRDD = tinyWords.zip(numRange)
print("zippedRDD result:")
for pair in zippedRDD.collect():
    print(pair)

## Task: Collocations (Bigrams)

**Background:**
Co-occurrence (or collocation) analysis evaluates and maps associative frequencies between words interacting across a large text corpus. A **bigram** is simply a pair of consecutive words.

**Part A: Basic Bigram Extraction**
Your first goal is to calculate the top 10 most frequent bigrams occurring throughout all the movie descriptions. You will execute this on the **Plot** textual entries defined within the Wikipedia dataset (`Column Index 7`). 
* Construct a `convertToWordPairs` function that uses map logic to split strings into array lists of 2-word combinations.
* Apply basic map-reduce logic resolving the counting frequencies metrics.

**Part B: Order-Independent Bigrams**
Currently, bigrams like "roman empire" and "empire roman" are counted as two completely separate bigrams due strictly to how they occur structurally in sentences. In Part B, recalculate the metrics but **count order-independent pairs identically** (so "roman empire" and "empire roman" both sum their values into a singular shared bucket!).
* *Hint: You will need to modify the helper function or your mappings so that every pair is alphabetized alphabetically prior to assignment.*

In [ ]:
import string
import nltk
from nltk.corpus import stopwords

# Download the stopwords corpus (usually required once per machine/cluster)
nltk.download('stopwords')

# Load the english stopwords into a set for fast lookup
stop_words = set(stopwords.words('english'))

In [ ]:
# --- PART A SOLUTION ---
import string

def convertToWordPairs(line):
    pairs = []
    # Strip punctuation and convert to lower case
    text = # TODO something
    
    # Split into words and only keep the ones that are NOT in our stopword set
    words = # TODO something using list comprehension
    
    for i in range(0, len(words) - 1):
        # TODO something
    return pairs

# 1. Collect corresponding plot values per individual instance mapped
plotsRDD = # TODO something

# Assuming plotsRDD is already defined...

bigramsRDD = # TODO something

print("PART A: Top 10 Order-Dependent Bigrams (No Stopwords):")
print(bigramsRDD.sortBy(lambda x: x[1], ascending=False).take(10))
# --- PART B SOLUTION ---

def convertToSortedWordPairs(line):
    pairs = []
    text = # TODO something
    
    # Remove stopwords here as well
    words = # TODO something with list comprehension
    
    for i in range(0, len(words) - 1):
        # Sort the two words alphabetically before combining them
        sorted_pair = # TODO something
        pairs.append("SOMETHING")
    return pairs

# Apply the same logic but mapped uniquely through the sorted helper
sortedBigramsRDD = # TODO something
print("\nPART B: Top 10 Order-Independent Bigrams (No Stopwords):")
print(sortedBigramsRDD.sortBy(lambda x: x[1], ascending=False).take(10))